# Data Preparation — Qwen3-TTS Fine-Tuning

Converts raw recordings into `finetune/train_with_codes.jsonl`.  
Run `03_finetune.ipynb` after this.

```
finetune/raw_recordings/   ← drop WAVs here
finetune/data/             ← chunks land here (auto-created)
```

## 1. Setup

In [1]:
import os, json, warnings, subprocess, sys, random
import torch, whisper, soundfile as sf
import IPython.display as ipd
from pathlib import Path
from pydub import AudioSegment
from pydub.silence import split_on_silence

warnings.filterwarnings("ignore")

os.environ["PATH"] = "C:/ffmpeg/bin;" + os.environ["PATH"]

# ── config ────────────────────────────────────────────────────────────────
RAW_DIR        = Path("finetune/raw_recordings")  # drop your WAV files here
DATA_DIR       = Path("finetune/data")
REF_AUDIO      = Path("audio/laxmikant_en.wav")
OUT_JSONL      = Path("finetune/train_raw.jsonl")
OUT_CODES      = Path("finetune/train_with_codes.jsonl")
TARGET_SR      = 24_000
MIN_DURATION_S = 2.0
MAX_DURATION_S = 15.0
# ──────────────────────────────────────────────────────────────────────────

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Check Raw Recordings

In [2]:
raw_files = sorted(RAW_DIR.glob("*.wav"))
for f in raw_files:
    dur = sf.info(str(f)).duration
    print(f.name, "-", round(dur / 60, 1), "min")
print("Total files:", len(raw_files))

sample_audio.wav - 7.4 min
Total files: 1


## 3. Split into Chunks

In [3]:
MIN_SILENCE_MS    = 600
SILENCE_THRESH_DB = -40
KEEP_SILENCE_MS   = 100

utterances = []
for rec in raw_files:
    seg = AudioSegment.from_wav(str(rec))
    seg = seg.set_frame_rate(TARGET_SR)
    seg = seg.set_channels(1)
    chunks = split_on_silence(seg, min_silence_len=MIN_SILENCE_MS,
                               silence_thresh=SILENCE_THRESH_DB,
                               keep_silence=KEEP_SILENCE_MS)
    for i, chunk in enumerate(chunks):
        dur = len(chunk) / 1000
        if dur < MIN_DURATION_S or dur > MAX_DURATION_S:
            continue
        out = DATA_DIR / f"{rec.stem[:30]}_{i:04d}.wav"
        chunk.export(str(out), format="wav")
        utterances.append((out, dur))

print("Chunks:", len(utterances))

Chunks: 80


## 4. Transcribe Chunks with Whisper large-v3

In [4]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

whisper_model = whisper.load_model("large-v3", device=device)

kept = []
for i, (wav_path, dur) in enumerate(utterances, 1):
    audio = whisper.load_audio(str(wav_path))
    result = whisper_model.transcribe(audio, language="en",
                                      beam_size=5, temperature=0.0,
                                      condition_on_previous_text=False)
    text = result["text"].strip()
    if len(text) < 5:
        wav_path.unlink()
        continue
    kept.append((wav_path, dur, text))
    if i % 200 == 0:
        print("Progress:", i, "/", len(utterances))

utterances = kept
print("Kept:", len(utterances))

Kept: 80


## 5. Review Samples (spot-check 10 random)

In [5]:
samples = random.sample(utterances, min(10, len(utterances)))
for path, dur, text in samples:
    data, sr = sf.read(str(path))
    print(path.name, "-", round(dur, 1), "s")
    print(" ", text)
    ipd.display(ipd.Audio(data, rate=sr))

sample_audio_0087.wav - 2.5 s
  The test of understanding is teaching.


sample_audio_0023.wav - 5.8 s
  The gap between what you meant and what you said is where most bugs live.


sample_audio_0079.wav - 5.1 s
  The people who think most clearly are usually the people who read most widely.


sample_audio_0066.wav - 5.2 s
  But the real story is always in what did not work and why.


sample_audio_0068.wav - 3.9 s
  Failure is not a positive of success, it is a part of the path.


sample_audio_0100.wav - 2.3 s
  Give yourself time, be patient.


sample_audio_0016.wav - 2.0 s
  that feeling never really goes out.


sample_audio_0028.wav - 2.1 s
  Let me tell you about the basics.


sample_audio_0007.wav - 3.8 s
  It is ability to break a problem into smaller pieces.


sample_audio_0017.wav - 5.1 s
  Every time something works, there is still that small moment of satisfaction.


## 6. Write train_raw.jsonl

In [6]:
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for path, dur, text in utterances:
        f.write(json.dumps({"audio": str(path.resolve()), "text": text,
                            "ref_audio": str(REF_AUDIO.resolve())}, ensure_ascii=False) + "\n")

## 7. Extract Audio Codes

In [7]:
ft_scripts = Path("finetune/scripts")
if not ft_scripts.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/QwenLM/Qwen3-TTS", str(ft_scripts)], check=True)
    subprocess.run(["git", "-C", str(ft_scripts), "sparse-checkout", "set", "finetuning"], check=True)

result = subprocess.run(
    [sys.executable, str(ft_scripts / "finetuning" / "prepare_data.py"),
     "--device",               "cuda:0",
     "--tokenizer_model_path", "Qwen/Qwen3-TTS-Tokenizer-12Hz",
     "--input_jsonl",          str(OUT_JSONL.resolve()),
     "--output_jsonl",         str(OUT_CODES.resolve())],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr[-2000:])
else:
    print("Entries:", len(OUT_CODES.read_text().splitlines()))

Entries: 80
